# Session 7: Functions, Parameters, Return Values and Lambda

**Python Fundamentals | Session 7 | Learning Assistant 7.0**

This guided notebook develops function interface design, argument binding, return contracts, state management, closures, higher-order functions, decomposition and verification. It assumes familiarity with control flow, loops, collections and comprehensions. Examples use British English, Python 3.10+ syntax and the standard library only; they run in Google Colab without files, network access or package installation.

The updated `Session_07_Advanced_Functions_Expanded.pptx` was not present in the workspace when this notebook was revised. The sequence therefore follows the requested topics. Each section is organised as **Concept → Predict → Run → Experiment → Debug → Reflect**. Prediction prompts place expected results after the question; reveal them only after making your own prediction. Invalid calls are explained in comments or Markdown rather than executed.


## Learning route and study data

We will evolve the Learning Assistant from a single script into a set of small, testable functions. The data represents topics and study durations in minutes. Calculations return values; only the final presentation step prints them. Later sections revisit the same data to demonstrate closures, sorting, aggregation and tests.

**Slide → Notebook → Verification → Next slide:** follow the numbered sections in sequence. At each step, run the examples, change at least one value, then use the assertions to check the contract.


## 1. Function interfaces: parameters, arguments and binding

**Learning sequence:** Concept → Predict → Run → Experiment → Debug → Reflect.


### 1.1 Parameters are names; arguments are supplied values

**Concept.** A parameter is written in a `def` statement; an argument is supplied by a caller. Python binds positional arguments from left to right. A clear function interface tells callers what information is required and what each value means.

**Predict.** Before running the code, calculate the total for each learner. Which name in the definition is the parameter, and which values in the calls are arguments?

<details><summary>Expected result (reveal after predicting)</summary>

The totals are `75` and `50`. `durations` is the parameter; each list supplied in a call is an argument.
</details>

In [ ]:
def calculate_total(durations):
    return sum(durations)


learner_a = [30, 45]
learner_b = [20, 30]
print(calculate_total(learner_a))
print(calculate_total(learner_b))
assert calculate_total(learner_a) == 75
assert calculate_total([]) == 0

### 1.2 Positional, keyword and mixed calls

**Concept.** Positional arguments are matched by order. Keyword arguments name a parameter, so their order can differ from the definition. A call may start with positional arguments and then use keywords; it cannot put a positional argument after a keyword argument.

**Predict.** All three valid calls below describe the same 45-minute threshold. Which call do you find easiest to scan?

<details><summary>Expected result (reveal after predicting)</summary>

Each call returns `[30, 45, 60]`. The keyword form makes the threshold's meaning explicit.
</details>

In [ ]:
def filter_sessions(durations, minimum_duration):
    return [
        duration
        for duration in durations
        if duration >= minimum_duration
    ]


sessions = [15, 30, 45, 60]
print(filter_sessions(sessions, 30))
print(filter_sessions(durations=sessions, minimum_duration=30))
print(filter_sessions(sessions, minimum_duration=30))
assert filter_sessions(sessions, 30) == [30, 45, 60]

### 1.3 Required parameters and argument binding

**Concept.** Unless a default is supplied, every parameter is required. Calls bind values to parameters before the function body starts. Python rejects a call if a required parameter is missing, a keyword is unknown, or one parameter receives two values.

**Predict.** Which calls in the comments are invalid, and why? They are intentionally shown as comments: running them would stop the cell with a `TypeError`.

<details><summary>Expected result (reveal after predicting)</summary>

The valid calls provide both required values. The commented calls respectively put a positional value after a keyword, bind `durations` twice, omit `minimum_duration`, or use an unexpected keyword.
</details>

In [ ]:
def describe_filter(durations, minimum_duration):
    return f"{len(durations)} sessions; threshold {minimum_duration} minutes"


print(describe_filter([25, 30], 30))
print(describe_filter(durations=[25, 30], minimum_duration=30))

# Invalid examples: inspect the binding problem without executing the call.
# describe_filter(durations=[25, 30], 30)  # positional after keyword
# describe_filter([25, 30], durations=[10], minimum_duration=30)  # duplicate
# describe_filter([25, 30])  # missing required argument
# describe_filter([25, 30], threshold=30)  # unexpected keyword

### 1.4 Definition ordering rules

**Concept.** Required parameters must precede parameters with defaults. Otherwise a later required parameter could not be distinguished from an omitted optional one. Positional-or-keyword parameters may be supplied positionally or by name, so choose names and defaults that make either form understandable.

**Predict.** Which default is used when the caller omits `minimum_duration`? Why is the invalid definition commented out?

<details><summary>Expected result (reveal after predicting)</summary>

The omitted threshold is `30`; the returned list is `[30, 45]`. `def invalid(required=1, also_required)` is a syntax error because a required parameter follows a defaulted parameter.
</details>

In [ ]:
def focused_sessions(durations, minimum_duration=30):
    return [value for value in durations if value >= minimum_duration]


# Invalid definition: it is not legal to put a required parameter after a
# parameter with a default.
# def invalid(required=1, also_required):
#     return required, also_required

print(focused_sessions([15, 30, 45]))
print(focused_sessions([15, 30, 45], 45))
assert focused_sessions([15, 30, 45]) == [30, 45]

### 1.5 Names make calls safer

**Experiment.** Compare a compact positional call with a keyword call. Rename the keyword in a scratch copy of the definition and notice that callers must be updated too: parameter names are part of the interface when callers use keywords.

**Debug.** A common error is swapping two same-typed positional arguments. The code runs, but the result is wrong; keyword calls make intent visible.

<details><summary>Expected result (reveal after predicting)</summary>

The positional call swaps the learner and topic names but still selects `[45, 60]`. The keyword call binds both names correctly.
</details>

In [ ]:
def learner_sessions(learner_name, topic, durations, minimum_duration):
    kept = [d for d in durations if d >= minimum_duration]
    return learner_name, topic, kept


study_data = [15, 30, 45, 60]
wrong_binding = learner_sessions("Python", "Aisha", study_data, 45)
clear_binding = learner_sessions(
    learner_name="Aisha",
    topic="Python",
    durations=study_data,
    minimum_duration=45,
)
print(wrong_binding)
print(clear_binding)
assert clear_binding == ("Aisha", "Python", [45, 60])

### 1.6 Verify the interface with realistic activity data

**Experiment.** Change the threshold and test the empty-list boundary. A reusable function should behave the same way regardless of which learner's data it receives.

**Reflect.** Which arguments are essential to this operation? When does a keyword make a call clearer than positional order?

<details><summary>Expected result (reveal after predicting)</summary>

At 30 minutes the selected study sessions are `[30, 45, 60]`; at 50 minutes only `[60]` remains. An empty list produces an empty result.
</details>

In [ ]:
weekly_study = {
    "Aisha": [15, 30, 45, 60],
    "Ben": [25, 50],
}
for learner, durations in weekly_study.items():
    selected = filter_sessions(
        durations=durations,
        minimum_duration=30,
    )
    print(learner, selected)

assert filter_sessions([], minimum_duration=30) == []
assert filter_sessions([30], minimum_duration=30) == [30]

## 2. Defaults, evaluation time and mutable state

**Learning sequence:** Concept → Predict → Run → Experiment → Debug → Reflect.


### 2.1 Default expressions are evaluated once

**Concept.** Python evaluates a default expression when it executes the `def` statement, not afresh on every call. Immutable defaults such as numbers, strings and `None` are usually safe. A default is therefore a stored object attached to the function.

**Predict.** Do both calls use the same threshold? What changes if the caller explicitly supplies `45`?

<details><summary>Expected result (reveal after predicting)</summary>

The first two calls use the same default threshold, `30`, and return `[30, 45]`. The explicit threshold returns `[45]`. Changing the source variable after defining a function does not change its captured default.
</details>

In [ ]:
def filter_at_default(durations, minimum_duration=30):
    return [d for d in durations if d >= minimum_duration]


week = [15, 30, 45]
print(filter_at_default(week))
print(filter_at_default(week))
print(filter_at_default(week, minimum_duration=45))
assert filter_at_default(week) == [30, 45]

configured_minimum = 30


def uses_definition_time_default(durations, minimum=configured_minimum):
    return [value for value in durations if value >= minimum]


configured_minimum = 60
assert uses_definition_time_default([30, 60]) == [30, 60]

### 2.2 The mutable-default trap

**Concept.** A list or dictionary default is created once and reused. Mutating it in one call changes what a later call sees. This is not a special list rule: the function keeps a reference to the same default object.

**Predict.** What will the second call display? The function is deliberately faulty, and the assertions reproduce the shared-state behaviour.

<details><summary>Expected result (reveal after predicting)</summary>

The first result is `['Python']`; the second is `['Python', 'Loops']`. Both calls appended to the same list.
</details>

In [ ]:
def faulty_add_topic(topic, topics=[]):
    topics.append(topic)
    return topics


first_topics = faulty_add_topic("Python")
second_topics = faulty_add_topic("Loops")
print(first_topics)
print(second_topics)
assert first_topics is second_topics
assert second_topics == ["Python", "Loops"]

### 2.3 Repair: use a `None` sentinel

**Concept.** Use `None` as the default, then create a fresh list inside the function. Test with `is None`, not `if not topics`: an explicitly supplied empty list is a real object the caller may want the function to fill.

**Predict.** Does each omitted argument get an independent list? Does passing `[]` preserve that exact list for the caller?

<details><summary>Expected result (reveal after predicting)</summary>

The two omitted-argument calls each return a one-item list. The explicitly supplied empty list is updated to `['Data']` and is still the same object.
</details>

In [ ]:
def add_topic(topic, topics=None):
    if topics is None:
        topics = []
    topics.append(topic)
    return topics


first = add_topic("Python")
second = add_topic("Loops")
provided = []
returned = add_topic("Data", provided)
print(first, second, provided)
assert first == ["Python"] and second == ["Loops"]
assert returned is provided and provided == ["Data"]

### 2.4 Why truthiness is not a substitute

**Debug.** The faulty version replaces any false-valued collection with a new list. That silently ignores the caller's empty list. An empty collection is false in a Boolean context, but emptiness does not mean “argument not supplied”.

**Predict.** Which list will contain `"Algorithms"` after the faulty call, and which list will contain it after the corrected call?

<details><summary>Expected result (reveal after predicting)</summary>

The faulty call leaves `caller_list` empty and returns another list. The `is None` version updates the caller's list.
</details>

In [ ]:
def faulty_add_with_truthiness(topic, topics=None):
    if not topics:
        topics = []
    topics.append(topic)
    return topics


caller_list = []
faulty_result = faulty_add_with_truthiness("Algorithms", caller_list)
assert caller_list == []
assert faulty_result == ["Algorithms"]

caller_list = []
correct_result = add_topic("Algorithms", caller_list)
assert caller_list == ["Algorithms"]
assert correct_result is caller_list
print("The sentinel pattern preserves an explicitly supplied empty list.")

### 2.5 Intentional persistence is different from an accidental default

**Concept.** Persistent state can be a deliberate feature, but it should be explicit and encapsulated. A closure (introduced in Section 6) can own a private collection that intentionally persists between calls. Do not rely on a mutable default to create this behaviour accidentally.

**Experiment.** Add a topic twice. The closure intentionally remembers it; creating a second recorder starts with independent state.

<details><summary>Expected result (reveal after predicting)</summary>

The first recorder remembers both topics and its repeated `Loops` entry. The second recorder remembers only `Functions`.
</details>

In [ ]:
def make_topic_recorder():
    recorded = []

    def record(topic):
        recorded.append(topic)
        return recorded.copy()

    return record


record_for_aisha = make_topic_recorder()
record_for_ben = make_topic_recorder()
print(record_for_aisha("Python"))
print(record_for_aisha("Loops"))
print(record_for_aisha("Loops"))
aisha_topics = record_for_aisha("Loops")
ben_topics = record_for_ben("Functions")
print(aisha_topics)
print(ben_topics)
assert aisha_topics == ["Python", "Loops", "Loops", "Loops"]
assert ben_topics == ["Functions"]

### 2.6 Confirm repeated-call behaviour

**Debug and reflect.** A faulty default may appear correct in a one-call demonstration. Test two calls and compare identities as well as values. Use the `None` pattern for caller-owned input, and use an explicit closure or object only when persistence is part of the documented contract.

<details><summary>Expected result (reveal after predicting)</summary>

The safe function returns separate lists on separate calls. The empty-list test confirms that a caller-provided list remains usable.
</details>

In [ ]:
def collect_activity_label(label, labels=None):
    if labels is None:
        labels = []
    labels.append(label)
    return labels


call_one = collect_activity_label("Reading")
call_two = collect_activity_label("Practice")
explicit_empty = []
collect_activity_label("Review", explicit_empty)
assert call_one == ["Reading"]
assert call_two == ["Practice"]
assert call_one is not call_two
assert explicit_empty == ["Review"]
print("Independent calls and explicit empty input are verified.")

## 3. Positional-only, positional-or-keyword and keyword-only

**Learning sequence:** Concept → Predict → Run → Experiment → Debug → Reflect.


### 3.1 Read a signature from left to right

**Concept.** In `def f(a, /, b, *, c)`, `a` is positional-only, `b` is positional-or-keyword, and `c` is keyword-only. `/` ends the positional-only region; `*` starts the keyword-only region. These markers control how callers may bind values.

**Predict.** Which calls below are valid? Why is a unit forced to be a keyword?

<details><summary>Expected result (reveal after predicting)</summary>

The positional call and the mixed call are valid. The call that names `minutes` is invalid because `minutes` appears before `/`.
</details>

In [ ]:
def convert_duration(minutes, /, unit="hours", *, precision=2):
    if unit == "hours":
        converted = minutes / 60
    elif unit == "seconds":
        converted = minutes * 60
    else:
        raise ValueError("unit must be 'hours' or 'seconds'")
    return round(converted, precision)


print(convert_duration(90))
print(convert_duration(90, "hours", precision=1))
print(convert_duration(90, unit="seconds", precision=0))
# Invalid: convert_duration(minutes=90)
assert convert_duration(90, precision=1) == 1.5

### 3.2 Configurable study threshold and unit

**Concept.** A positional-only value can be suitable when its meaning is obvious and its name need not become a public keyword. Keyword-only configuration is especially useful for thresholds, units and rounding options: callers must label choices rather than rely on a fragile sequence of positional values.

**Predict.** How many sessions meet the 30-minute threshold? Does a unit label change the calculation?

<details><summary>Expected result (reveal after predicting)</summary>

A 30-minute session meets the inclusive threshold; two sessions qualify. The unit is included in the formatted result.
</details>

In [ ]:
def count_qualifying_sessions(durations, /, *, minimum=30):
    return sum(duration >= minimum for duration in durations)


def describe_duration(minutes, /, *, unit="minutes"):
    return f"{minutes} {unit}"


study_sessions = [15, 30, 45]
print(count_qualifying_sessions(study_sessions, minimum=30))
print(describe_duration(45))
print(describe_duration(45, unit="minutes"))
assert count_qualifying_sessions(study_sessions, minimum=30) == 2

### 3.3 Report formatting as keyword-only configuration

**Concept.** Formatting options often have sensible defaults and many possible combinations. Making them keyword-only prevents a call such as `build_report(data, 30, 2, False)` from hiding the meaning of `2` and `False`.

**Predict.** Which sessions appear at the exact threshold? What changes when the caller selects hours and one decimal place?

<details><summary>Expected result (reveal after predicting)</summary>

At a 30-minute minimum, `[30, 60]` qualifies. In hours with one decimal place, the result is `[0.5, 1.0]`.
</details>

In [ ]:
def format_sessions(
    durations,
    /,
    *,
    minimum=30,
    unit="minutes",
    precision=1,
):
    selected = [value for value in durations if value >= minimum]
    if unit == "hours":
        selected = [value / 60 for value in selected]
    elif unit != "minutes":
        raise ValueError("unit must be 'minutes' or 'hours'")
    return [round(value, precision) for value in selected]


week = [15, 30, 60]
print(format_sessions(week, minimum=30))
print(format_sessions(week, minimum=30, unit="hours", precision=1))
assert format_sessions(week, minimum=30, unit="hours") == [0.5, 1.0]

### 3.4 Invalid calls are interface mistakes

**Debug.** Read the comments and diagnose the binding rule without executing an invalid call. A positional-only parameter cannot be passed by name; a keyword-only parameter cannot be supplied positionally. In the valid call, the flexible parameter may be given either way.

<details><summary>Expected result (reveal after predicting)</summary>

`analyse([30], minimum=30)` is valid. The commented calls violate `/` or `*` restrictions.
</details>

In [ ]:
def analyse(durations, /, threshold=30, *, unit="minutes"):
    return {
        "count": sum(value >= threshold for value in durations),
        "unit": unit,
    }


print(analyse([20, 30, 45], threshold=30, unit="minutes"))
# Invalid: analyse(durations=[20, 30], threshold=30)
# Invalid: analyse([20, 30], 30, "minutes")
assert analyse([], unit="hours") == {"count": 0, "unit": "hours"}

### 3.5 Choose restrictions for clarity, not fashion

**Experiment.** Try different units and thresholds. Restrict parameters only when it improves the interface: callers should be able to guess a valid call from the signature and its documentation.

**Reflect.** Which options are essential? Which are configuration? How do keyword-only settings reduce accidental misuse?

<details><summary>Expected result (reveal after predicting)</summary>

The chosen report records one qualifying activity and a total of `60 minutes` at the 60-minute threshold.
</details>

In [ ]:
def activity_report(durations, /, *, threshold=30, unit="minutes"):
    qualifying = [d for d in durations if d >= threshold]
    total = sum(qualifying)
    if unit == "hours":
        total /= 60
    elif unit != "minutes":
        raise ValueError("unit must be 'minutes' or 'hours'")
    return {
        "qualifying_count": len(qualifying),
        "total": round(total, 1),
        "unit": unit,
    }


report = activity_report([30, 60], threshold=60, unit="minutes")
print(report)
assert report == {"qualifying_count": 1, "total": 60, "unit": "minutes"}

### 3.6 A boundary test for a configurable threshold

**Experiment.** Compare values just below, equal to and above the threshold. Then test zero and an empty collection. Explicitly deciding the equality rule is part of the function contract.

**Reflect.** Would a caller reasonably expect the exact threshold to qualify? Record that decision in the name, documentation or tests.

<details><summary>Expected result (reveal after predicting)</summary>

For threshold `30`, `29` is excluded and `30` and `31` are included. Empty input returns zero.
</details>

In [ ]:
threshold_cases = [29, 30, 31]
assert count_qualifying_sessions(threshold_cases, minimum=30) == 2
assert count_qualifying_sessions([], minimum=30) == 0
assert count_qualifying_sessions([0], minimum=0) == 1
print(activity_report([], threshold=0))
print("Threshold boundary checks passed.")

## 4. Variable-length arguments and unpacking

**Learning sequence:** Concept → Predict → Run → Experiment → Debug → Reflect.


### 4.1 `*args` collects extra positional arguments

**Concept.** In a function definition, `*args` collects any remaining positional arguments into a tuple. The name `args` is a convention, not a keyword. Fixed parameters can come first, followed by the variadic positional parameter.

**Predict.** What are `topic`, `durations` and the type of `durations` when the function is called with three duration values?

<details><summary>Expected result (reveal after predicting)</summary>

`topic` is `"Python"`, `durations` is `(25, 40, 55)`, and its type is `tuple`.
</details>

In [ ]:
def summarize_topic(topic, *durations):
    return {
        "topic": topic,
        "durations": durations,
        "type": type(durations).__name__,
        "total": sum(durations),
    }


summary = summarize_topic("Python", 25, 40, 55)
print(summary)
assert summary["durations"] == (25, 40, 55)
assert summary["total"] == 120

### 4.2 Zero or many durations

**Concept.** A variadic parameter may receive zero values. Its tuple is then empty, so code should define what that means. Here, `sum(())` is zero and the average is deliberately `None` because there is no meaningful average.

**Predict.** What does the function return for no durations and for two durations?

<details><summary>Expected result (reveal after predicting)</summary>

No values produce `{"total": 0, "average": None}`. Durations `30` and `60` produce a total of `90` and an average of `45`.
</details>

In [ ]:
def duration_statistics(*durations):
    total = sum(durations)
    average = total / len(durations) if durations else None
    return {"count": len(durations), "total": total, "average": average}


print(duration_statistics())
print(duration_statistics(30, 60))
assert duration_statistics()["average"] is None
assert duration_statistics(30, 60)["average"] == 45

### 4.3 `**kwargs` collects optional metadata

**Concept.** In a function definition, `**kwargs` collects extra keyword arguments into a dictionary. It is useful when a report can carry optional metadata, but do not use it to hide a poorly designed set of known parameters.

**Predict.** What type and contents will `metadata` have when a caller supplies a cohort and a week number?

<details><summary>Expected result (reveal after predicting)</summary>

It is a dictionary: `{'cohort': 'UK', 'week': 7}`.
</details>

In [ ]:
def build_metadata_report(title, **metadata):
    return {"title": title, "metadata": metadata}


metadata_report = build_metadata_report(
    "Weekly study",
    cohort="UK",
    week=7,
)
print(metadata_report)
assert metadata_report["metadata"] == {"cohort": "UK", "week": 7}

### 4.4 Combine fixed, variable and keyword-only parameters

**Concept.** A signature can combine a required fixed parameter, `*args`, and keyword-only options. Parameters after `*args` are keyword-only. The call remains readable because durations and report settings have distinct roles.

**Predict.** What is the returned total and unit? What happens when there are no durations?

<details><summary>Expected result (reveal after predicting)</summary>

The first report totals `90 minutes`; the no-duration report totals `0 minutes`.
</details>

In [ ]:
def build_duration_report(topic, *durations, unit="minutes"):
    return {
        "topic": topic,
        "durations": durations,
        "total": sum(durations),
        "unit": unit,
    }


print(build_duration_report("Functions", 30, 60, unit="minutes"))
print(build_duration_report("Reading", unit="minutes"))
assert build_duration_report("Reading")["total"] == 0

### 4.5 Definition unpacking is not call-site unpacking

**Concept.** `*args` in a definition collects values. At a call site, `*iterable` expands an iterable into positional arguments. Likewise, `**kwargs` collects keywords, whereas `**mapping` expands a mapping into named arguments.

**Predict.** How many items are in the returned tuple? What labels appear in the report?

<details><summary>Expected result (reveal after predicting)</summary>

The list and tuple expand to three values `(20, 30, 45)`. The report contains the `cohort` and `week` metadata.
</details>

In [ ]:
def inspect_call(*args, **kwargs):
    return args, kwargs


duration_list = [20, 30]
duration_tuple = (45,)
metadata_mapping = {"cohort": "UK", "week": 7}
collected = inspect_call(*duration_list, *duration_tuple, **metadata_mapping)
print(collected)
assert collected == ((20, 30, 45), {"cohort": "UK", "week": 7})

### 4.6 Conflicts and unexpected names

**Debug.** A parameter cannot receive two values. Expanding a mapping that repeats a keyword already supplied explicitly is invalid; so is supplying a key the function does not accept when no `**kwargs` collector exists. Keep invalid forms commented so the learning cell runs safely.

**Experiment.** Try list, tuple and dictionary unpacking in your own function call.

**Reflect.** When does accepting arbitrary keywords make an interface more useful, and when might it hide a typo?

<details><summary>Expected result (reveal after predicting)</summary>

The valid call combines two durations and metadata. The commented forms would raise `TypeError` because of a duplicate binding or an unexpected keyword.
</details>

In [ ]:
def report_for_topic(topic, *durations, **metadata):
    return topic, sum(durations), metadata


extra = {"week": 7}
print(report_for_topic("Loops", *[15, 30], **extra))
# Invalid: duplicate topic binding
# report_for_topic("Loops", topic="Functions")
# Invalid: unexpected keyword if there is no **metadata parameter
# def fixed_report(title):
#     return title
# fixed_report(title="Week 7", cohort="UK")
assert report_for_topic("Loops", *(), **{}) == ("Loops", 0, {})

## 5. Return values, contracts and data flow

**Learning sequence:** Concept → Predict → Run → Experiment → Debug → Reflect.


### 5.1 A function contract: inputs, output and behaviour

**Concept.** A contract states what inputs a function expects, what it returns, and whether it mutates input. Type hints document intended types but do not validate them at runtime. A docstring gives the caller a concise behavioural promise.

**Predict.** What should the total be for the two recorded durations? Does the function display anything itself?

<details><summary>Expected result (reveal after predicting)</summary>

The total is `75`. The function returns it; only the caller's `print` displays it.
</details>

In [ ]:
def calculate_total(durations: list[int | float]) -> int | float:
    '''Return the sum of study durations in minutes.'''
    return sum(durations)


activity_durations = [30.0, 45.0]
total_minutes = calculate_total(activity_durations)
print(total_minutes)
assert total_minutes == 75.0

### 5.2 Early returns make branches explicit

**Concept.** An early return handles a special case and ends that function call immediately. This can prevent invalid arithmetic and make the main path easier to read. Every path should honour the documented return contract.

**Predict.** What does the average function return for an empty list? What does it return for two durations?

<details><summary>Expected result (reveal after predicting)</summary>

The empty case returns `None`; `[30, 60]` returns `45.0`.
</details>

In [ ]:
def calculate_average(durations: list[float]) -> float | None:
    '''Return the average, or None when there are no durations.'''
    if not durations:
        return None
    return sum(durations) / len(durations)


print(calculate_average([]))
print(calculate_average([30.0, 60.0]))
assert calculate_average([]) is None
assert calculate_average([30.0, 60.0]) == 45.0

### 5.3 Multiple values and tuple unpacking

**Concept.** Returning comma-separated values creates a tuple. Tuple unpacking assigns its elements to caller names in order. This is useful for a small, fixed result; return a dictionary when named fields make a larger result clearer.

**Predict.** What values will `total`, `count` and `longest` receive for the activity list?

<details><summary>Expected result (reveal after predicting)</summary>

They receive `105`, `3` and `45`, respectively. The empty case is explicitly `(0, 0, 0)`.
</details>

In [ ]:
def session_summary(durations: list[int]) -> tuple[int, int, int]:
    '''Return total minutes, count and longest session.'''
    if not durations:
        return 0, 0, 0
    return sum(durations), len(durations), max(durations)


total, count, longest = session_summary([30, 45, 30])
print(total, count, longest)
assert (total, count, longest) == (105, 3, 45)
assert session_summary([]) == (0, 0, 0)

### 5.4 Returning collections and preserving inputs

**Concept.** A transformation can build and return a new collection rather than mutate the caller's data. This makes data flow easier to verify: the original records remain available for other calculations.

**Predict.** Which durations meet the threshold? Is the returned list the same object as the input?

<details><summary>Expected result (reveal after predicting)</summary>

The selected values are `[30, 45]`. The returned list is a new object; the original list remains unchanged.
</details>

In [ ]:
def filter_sessions(
    durations: list[int],
    minimum_duration: int,
) -> list[int]:
    '''Return sessions meeting an inclusive minimum; preserve the input.'''
    return [duration for duration in durations if duration >= minimum_duration]


original = [15, 30, 45]
selected = filter_sessions(original, 30)
print(selected)
assert selected == [30, 45]
assert selected is not original
assert original == [15, 30, 45]

### 5.5 Return a conversion; do not print from the calculation

**Concept.** Returning values separates calculation from input/output. A returned list can be tested, formatted or used in another function. An implicit return is `None`; this is appropriate when a function's contract is to perform an action, but a missing return is a defect when callers need a result.

**Debug.** Predict what the faulty version returns, then compare it with the corrected conversion.

<details><summary>Expected result (reveal after predicting)</summary>

The print-only version returns `None`. The correct conversion returns `[0.5, 1.0]` hours for 30 and 60 minutes.
</details>

In [ ]:
def faulty_convert_to_hours(durations):
    print([duration / 60 for duration in durations])


def convert_to_hours(durations: list[float]) -> list[float]:
    '''Return a new list of durations converted from minutes to hours.'''
    return [duration / 60 for duration in durations]


faulty_result = faulty_convert_to_hours([30, 60])
correct_result = convert_to_hours([30, 60])
print("Faulty return:", faulty_result)
print("Correct return:", correct_result)
assert faulty_result is None
assert correct_result == [0.5, 1.0]

### 5.6 A report composes useful return values

**Experiment.** Combine filtering, total calculation and conversion. Keyword-only options make report configuration visible. The report function returns data; a separate presentation step decides how to display it.

**Reflect.** Can a caller save this report, test it or render it differently without changing the calculation?

<details><summary>Expected result (reveal after predicting)</summary>

The report includes a total of `150` minutes, focused sessions `[30, 45, 60]`, and the corresponding hours `[0.5, 0.75, 1.0]`.
</details>

In [ ]:
def build_report(
    durations: list[int],
    *,
    minimum_duration: int = 30,
) -> dict[str, object]:
    '''Return a study report without displaying or mutating input data.'''
    focused = filter_sessions(durations, minimum_duration)
    return {
        "total_minutes": calculate_total(durations),
        "focused_sessions": focused,
        "focused_hours": convert_to_hours(focused),
    }


week = [15, 30, 45, 60]
study_report = build_report(week, minimum_duration=30)
print(study_report)
assert study_report["total_minutes"] == 150
assert week == [15, 30, 45, 60]

## 6. Scope, nested functions and closures

**Learning sequence:** Concept → Predict → Run → Experiment → Debug → Reflect.


### 6.1 LEGB: how Python finds a name

**Concept.** Python searches for a referenced name in this order: **Local**, **Enclosing**, **Global**, **Built-in**. A local name belongs to the current function call; enclosing names belong to surrounding function scopes; global names belong to the module; built-ins include names such as `sum` and `len`.

**Predict.** Which `minimum` value is used by `qualifies`? Which scope provides `len`?

<details><summary>Expected result (reveal after predicting)</summary>

The local parameter `minimum` is `30`, so the session qualifies. `len` is found in the built-in scope.
</details>

In [ ]:
minimum = 999


def qualifies(duration, minimum):
    return duration >= minimum and len([duration]) == 1


print(qualifies(30, 30))
assert qualifies(30, 30) is True

### 6.2 A nested function can use an enclosing value

**Concept.** A nested function can read a name from its enclosing function's local scope. That is the **E** in LEGB. The value need not be global, and callers cannot accidentally change it by rebinding an unrelated global name.

**Predict.** What is the configured minimum? What happens to a duration exactly equal to it?

<details><summary>Expected result (reveal after predicting)</summary>

The enclosing minimum is `40`. Durations `40` and `50` are accepted.
</details>

In [ ]:
def make_duration_filter(minimum):
    def keep(duration):
        return duration >= minimum

    return keep


keep_40_or_more = make_duration_filter(40)
print([d for d in [20, 40, 50] if keep_40_or_more(d)])
assert keep_40_or_more(40)
assert not keep_40_or_more(39)

### 6.3 Closures retain enclosing variables

**Concept.** A closure is a function that retains access to names from its enclosing scope. The outer call can finish, yet the returned function can still use the configured minimum. Each call to the factory creates an independent configuration.

**Predict.** Do the 30-minute and 60-minute filters return the same selections?

<details><summary>Expected result (reveal after predicting)</summary>

No. The 30-minute filter keeps `[30, 45, 60]`; the 60-minute filter keeps `[60]`.
</details>

In [ ]:
keep_30 = make_duration_filter(30)
keep_60 = make_duration_filter(60)
sample_sessions = [15, 30, 45, 60]
selected_30 = [d for d in sample_sessions if keep_30(d)]
selected_60 = [d for d in sample_sessions if keep_60(d)]
print(selected_30, selected_60)
assert selected_30 == [30, 45, 60]
assert selected_60 == [60]

### 6.4 Closure or ordinary parameter?

**Concept.** A closure is useful when a configured function will be reused many times, such as a filter for one learner's threshold. A normal parameter is simpler for a one-off call or when the configuration changes frequently. Choose the simplest interface that communicates intent.

**Experiment.** Compare both forms. They implement the same inclusive rule but expose configuration differently.

<details><summary>Expected result (reveal after predicting)</summary>

Both forms return `[30, 45]` for the supplied data and threshold.
</details>

In [ ]:
def keep_at_least(duration, minimum):
    return duration >= minimum


configured_keep = make_duration_filter(30)
parameter_result = [
    d for d in [15, 30, 45] if keep_at_least(d, 30)
]
closure_result = [
    d for d in [15, 30, 45] if configured_keep(d)
]
print(parameter_result, closure_result)
assert parameter_result == closure_result == [30, 45]

### 6.5 `nonlocal` updates private enclosing state

**Concept.** Assignment inside a nested function normally creates a local name. `nonlocal` tells Python to rebind a name in the nearest enclosing function scope. This is suitable for small private state, such as a counter, when that state is part of the function's purpose.

**Predict.** What does the counter return on its first and second calls? Can code outside the closure directly access `count`?

<details><summary>Expected result (reveal after predicting)</summary>

The calls return `1` and `2`. `count` is local to the factory's enclosing scope and is not exposed as a global.
</details>

In [ ]:
def make_counter():
    count = 0

    def next_count():
        nonlocal count
        count += 1
        return count

    return next_count


activity_counter = make_counter()
print(activity_counter())
print(activity_counter())
assert activity_counter() == 3

### 6.6 Late binding in a loop and its repair

**Concept.** A closure created in a loop looks up the loop variable when it is later called, not when it is created. If all functions refer to the same changing variable, they can all observe its final value. A default argument captures the current value at function creation time.

**Predict.** What will the faulty functions return after the loop? How does the corrected version differ?

**Debug.** Identify when the loop's `threshold` is read in each version. The default argument saves that iteration's value.

**Reflect.** Where might a closure factory be more maintainable than building lambdas in a loop?

<details><summary>Expected result (reveal after predicting)</summary>

The faulty functions all return `False` for a 45-minute duration. The corrected functions return `[True, True, False]`.
</details>

In [ ]:
thresholds = [15, 30, 60]
late_filters = []
for threshold in thresholds:
    late_filters.append(lambda duration: duration >= threshold)

captured_filters = []
for threshold in thresholds:
    captured_filters.append(
        lambda duration, minimum=threshold: duration >= minimum
    )

print([f(45) for f in late_filters])
print([f(45) for f in captured_filters])
assert [f(45) for f in late_filters] == [False, False, False]
assert [f(45) for f in captured_filters] == [True, True, False]

## 7. Lambda expressions and higher-order functions

**Learning sequence:** Concept → Predict → Run → Experiment → Debug → Reflect.


### 7.1 Lambda syntax and a named equivalent

**Concept.** `lambda arguments: expression` creates a function whose body is one expression; the expression's value is returned automatically. A lambda has no statement suite, so multi-step behaviour, validation and documentation belong in a named `def`.

**Predict.** Do the named function and lambda return the same value for a 30-minute activity?

<details><summary>Expected result (reveal after predicting)</summary>

Both return `60`; this tiny operation is simple enough to compare directly.
</details>

In [ ]:
def double_minutes(minutes):
    return minutes * 2


double_with_lambda = lambda minutes: minutes * 2
print(double_minutes(30), double_with_lambda(30))
assert double_minutes(30) == double_with_lambda(30) == 60

### 7.2 Sort tuples by duration, then topic

**Concept.** Functions are values in Python. `sorted` is a higher-order function because it accepts another function as its `key`. A lambda is useful here: it gives a short, local sorting rule without introducing a one-use top-level name.

**Predict.** What is the order by duration, then alphabetically by topic for equal durations?

<details><summary>Expected result (reveal after predicting)</summary>

The order is `('Loops', 30)`, `('Python', 30)`, then `('Data', 45)`.
</details>

In [ ]:
topic_durations = [
    ("Python", 30),
    ("Data", 45),
    ("Loops", 30),
]
sorted_tuples = sorted(topic_durations, key=lambda item: (item[1], item[0]))
print(sorted_tuples)
assert sorted_tuples == [("Loops", 30), ("Python", 30), ("Data", 45)]

### 7.3 Sort dictionaries and records by multiple keys

**Concept.** A key can be a tuple of values, giving lexicographic sorting by multiple keys. `reverse=True` reverses the complete ordering; for more nuanced ascending/descending combinations, use a transformation in the key or sort in stable stages.

**Predict.** Which learner comes first by ascending total and ascending learner name for a tie?

<details><summary>Expected result (reveal after predicting)</summary>

The two records with 60 minutes are ordered Aisha, then Ben, followed by Zoe's 90-minute record.
</details>

In [ ]:
study_records = [
    {"learner": "Ben", "topic": "Functions", "minutes": 60},
    {"learner": "Aisha", "topic": "Loops", "minutes": 60},
    {"learner": "Zoe", "topic": "Collections", "minutes": 90},
]
by_minutes = sorted(
    study_records,
    key=lambda row: (row["minutes"], row["learner"]),
)
descending_minutes = sorted(
    study_records,
    key=lambda row: row["minutes"],
    reverse=True,
)
print(by_minutes)
print(descending_minutes)
assert by_minutes[0]["learner"] == "Aisha"
assert descending_minutes[0]["minutes"] == 90

### 7.4 Transform and select with `map` and `filter`

**Concept.** `map(function, iterable)` applies a function to each value; `filter(function, iterable)` keeps values for which the function is truthy. Both return iterators, so convert them to a list to inspect or reuse all results. Comprehensions are often clearer for simple transformations.

**Predict.** Which converted values are at least half an hour?

<details><summary>Expected result (reveal after predicting)</summary>

Hours are `[0.25, 0.5, 1.0]`; the selected values are `[0.5, 1.0]`.
</details>

In [ ]:
minutes = [15, 30, 60]
hours = list(map(lambda value: value / 60, minutes))
long_sessions = list(filter(lambda value: value >= 0.5, hours))
comprehension_hours = [value / 60 for value in minutes]
print(hours, long_sessions)
assert hours == comprehension_hours == [0.25, 0.5, 1.0]
assert long_sessions == [0.5, 1.0]

### 7.5 Lambda with dictionaries and custom transformations

**Experiment.** Sort topic totals in descending order, breaking ties alphabetically. Try changing the key to sort by the length of the topic name; the key may compute a comparison value without changing the original records.

**Predict.** Which topic is first in descending order of total?

<details><summary>Expected result (reveal after predicting)</summary>

Python is first with 120 minutes, followed by Loops at 60 and Testing at 30.
</details>

In [ ]:
topic_totals = {
    "Python": 120,
    "Loops": 60,
    "Testing": 30,
}
topic_rows = list(topic_totals.items())
ranked_topics = sorted(
    topic_rows,
    key=lambda item: (-item[1], item[0]),
)
by_name_length = sorted(topic_rows, key=lambda item: len(item[0]))
print(ranked_topics)
print(by_name_length)
assert ranked_topics[0] == ("Python", 120)

### 7.6 A lambda factory, and when to prefer `def`

**Concept.** A lambda can produce a short configured function, but a named factory is clearer when configuration or behaviour deserves explanation. Prefer a named function when it is reused, has a domain-specific name, needs a docstring, or contains more than a simple expression.

**Debug.** The one-expression lambda is short, but a more involved predicate needs a descriptive name. The factory below compares three alternatives without relying on hidden state.

**Reflect.** Which form would you keep in production, and what does the named version communicate?

<details><summary>Expected result (reveal after predicting)</summary>

Both configured predicates accept 30 and reject 15 for a 30-minute minimum.
</details>

In [ ]:
minimum = 30
compact_predicate = lambda duration: duration >= minimum


lambda_factory = lambda minimum_duration: (
    lambda duration: duration >= minimum_duration
)


def named_factory(minimum_duration):
    def keep(duration):
        return duration >= minimum_duration

    return keep


named_predicate = named_factory(minimum)
factory_predicate = lambda_factory(minimum)
print(compact_predicate(30), factory_predicate(15), named_predicate(15))
assert compact_predicate(30) == named_predicate(30) is True
assert factory_predicate(30) is True
assert compact_predicate(15) == named_predicate(15) is False

## 8. Decomposition and the Learning Assistant 7.0

**Learning sequence:** Concept → Predict → Run → Experiment → Debug → Reflect.


### 8.1 Define the data flow before writing the application

**Concept.** Decomposition separates collection, calculation, transformation, aggregation and presentation. The Learning Assistant 7.0 can use small functions with clear contracts: each accepts data and returns a result. Input/output belongs at the boundary, not inside reusable calculations.

**Predict.** What should the collector return for two activities? Does it need to call `input()` to be testable?

<details><summary>Expected result (reveal after predicting)</summary>

It returns the supplied records. Passing records in makes the collector logic deterministic and easy to test without interactive input.
</details>

In [ ]:
def collect_activities(records):
    '''Return a copy of the supplied activity records.'''
    return [record.copy() for record in records]


sample_activities = [
    {"topic": "Python", "minutes": 30},
    {"topic": "Functions", "minutes": 45},
]
collected = collect_activities(sample_activities)
print(collected)
assert collected == sample_activities
assert collected is not sample_activities

### 8.2 Calculate totals and filter sessions independently

**Concept.** One function calculates all recorded minutes; another applies the focused-session rule. They can be verified separately, so a bug in one is easier to localise than in a single large report function.

**Predict.** What is the total, and which sessions meet the 30-minute threshold?

<details><summary>Expected result (reveal after predicting)</summary>

The total is `75` minutes. The selected sessions are 30 and 45 minutes.
</details>

In [ ]:
def total_activity_minutes(activities):
    return sum(item["minutes"] for item in activities)


def select_focused_sessions(activities, *, minimum=30):
    return [
        item.copy()
        for item in activities
        if item["minutes"] >= minimum
    ]


records = collect_activities(sample_activities)
focused = select_focused_sessions(records, minimum=30)
print(total_activity_minutes(records), focused)
assert total_activity_minutes(records) == 75
assert [item["minutes"] for item in focused] == [30, 45]

### 8.3 Convert units and accumulate topic totals

**Concept.** Conversion and aggregation are distinct transformations. Topic aggregation combines repeated topics using a dictionary; it should not mutate the source records. Keyword-only options clarify configurable units and thresholds.

**Predict.** What are the topic totals for the activity data? What are 30 and 45 minutes in hours?

<details><summary>Expected result (reveal after predicting)</summary>

Python totals 50 minutes, Functions totals 45, and the converted values are 0.5 and 0.75 hours.
</details>

In [ ]:
def convert_activity_units(minutes, /, *, unit="hours"):
    if unit == "hours":
        return minutes / 60
    if unit == "minutes":
        return minutes
    raise ValueError("unit must be 'minutes' or 'hours'")


def accumulate_topic_totals(activities):
    totals = {}
    for item in activities:
        topic = item["topic"]
        totals[topic] = totals.get(topic, 0) + item["minutes"]
    return totals


activity_log = [
    {"topic": "Python", "minutes": 30},
    {"topic": "Functions", "minutes": 45},
    {"topic": "Python", "minutes": 20},
]
print(accumulate_topic_totals(activity_log))
print(convert_activity_units(30), convert_activity_units(45))
assert accumulate_topic_totals(activity_log) == {"Python": 50, "Functions": 45}

### 8.4 Build a report from smaller functions

**Concept.** A report function coordinates other pure calculations and returns structured data. It does not print, ask for input or modify records. This is the composition step: small tested functions collaborate through their return values.

**Predict.** How many focused sessions are there? Does the report keep all-study total separate from focused-session total?

<details><summary>Expected result (reveal after predicting)</summary>

There are two focused sessions. The report contains all-study total `95`, focused total `75`, and per-topic totals.
</details>

In [ ]:
def build_learning_report(activities, *, minimum=30, unit="hours"):
    focused = select_focused_sessions(activities, minimum=minimum)
    focused_minutes = total_activity_minutes(focused)
    return {
        "activity_count": len(activities),
        "total_minutes": total_activity_minutes(activities),
        "focused_count": len(focused),
        "focused_minutes": focused_minutes,
        "focused_total": convert_activity_units(focused_minutes, unit=unit),
        "topic_totals": accumulate_topic_totals(activities),
    }


records = collect_activities([
    {"topic": "Python", "minutes": 30},
    {"topic": "Functions", "minutes": 45},
    {"topic": "Reading", "minutes": 20},
])
report = build_learning_report(records, minimum=30, unit="minutes")
print(report)
assert report["total_minutes"] == 95
assert report["focused_minutes"] == 75
assert report["focused_count"] == 2

### 8.5 Keep presentation separate from calculation

**Concept.** The same returned report can be shown as console text, a notebook display or a future user interface. A presentation function may print; the calculation functions should return values. This boundary also allows tests to assert exact report data without capturing printed output.

**Experiment.** Change the display function's formatting without touching report calculations.

<details><summary>Expected result (reveal after predicting)</summary>

The display function prints three lines. The report remains a dictionary and is unchanged by displaying it.
</details>

In [ ]:
def display_learning_report(report):
    print(f"Activities: {report['activity_count']}")
    print(f"Total: {report['total_minutes']} minutes")
    print(f"Focused: {report['focused_count']} sessions")


before_display = report.copy()
display_learning_report(report)
assert report == before_display
assert report["topic_totals"] == {
    "Python": 30,
    "Functions": 45,
    "Reading": 20,
}

### 8.6 Test each function before integration

**Verification.** Assert each contract independently, then check an integrated boundary case. Empty activity data should produce zero counts and totals, an empty topic mapping, and no focused sessions. No file, network or interactive input is needed.

**Reflect.** Which function would you change if the definition of a focused session changed? Which functions should remain unaffected?

**Debug.** If the empty-data assertion fails, test each component's empty-input contract before inspecting the final report.

<details><summary>Expected result (reveal after predicting)</summary>

An empty report contains zero activity and focused counts, zero totals, and an empty topic-total dictionary.
</details>

In [ ]:
assert collect_activities([]) == []
assert total_activity_minutes([]) == 0
assert select_focused_sessions([], minimum=30) == []
assert accumulate_topic_totals([]) == {}
empty_report = build_learning_report([], minimum=30, unit="hours")
assert empty_report == {
    "activity_count": 0,
    "total_minutes": 0,
    "focused_count": 0,
    "focused_minutes": 0,
    "focused_total": 0,
    "topic_totals": {},
}
print("Component and integration checks passed.")

## 9. Debugging, boundary tests and engineering verification

**Learning sequence:** Concept → Predict → Run → Experiment → Debug → Reflect.


### 9.1 Debug argument binding without hiding the rule

**Concept.** A function can run successfully yet do the wrong thing when positional values are swapped. Use names and a realistic example to see how values bind. Python reports duplicate, missing and unexpected arguments before entering the function body; comments are a safe way to study invalid calls.

**Predict.** What does the confusing positional call bind to each parameter? Which call communicates intent better?

<details><summary>Expected result (reveal after predicting)</summary>

The first call binds `"Aisha"` as the topic and `"Python"` as the learner, reversing their meaning. The keyword call binds correctly.
</details>

In [ ]:
def label_activity(learner, topic):
    return f"{learner} studied {topic}"


confusing = label_activity("Python", "Aisha")
clear = label_activity(learner="Aisha", topic="Python")
print(confusing)
print(clear)
# Invalid: label_activity(learner="Aisha", "Python")
# Invalid: label_activity("Aisha", learner="Ben")
# Invalid: label_activity("Aisha", course="Python")
assert clear == 'Aisha studied Python'

### 9.2 Debug a missing return value

**Concept.** A function that reaches its end without `return` produces `None`, even if it calculated a local variable or printed a value. Test the returned value rather than judging by console output alone.

**Predict.** What does the faulty summary return? How does the corrected version express the output contract?

<details><summary>Expected result (reveal after predicting)</summary>

The faulty function returns `None`. The repaired function returns the numeric total `75`.
</details>

In [ ]:
def faulty_total(durations):
    total = sum(durations)
    print("Calculated:", total)


def repaired_total(durations):
    return sum(durations)


missing_result = faulty_total([30, 45])
repaired_result = repaired_total([30, 45])
print("Faulty result:", missing_result)
assert missing_result is None
assert repaired_result == 75

### 9.3 Debug unintended mutation

**Concept.** Assignment does not copy a list; both names then refer to the same object. Mutating through either name changes that object. If a function promises to preserve caller data, build a new collection or make an intentional copy before mutation.

**Predict.** Does the faulty function change `original_durations`? Does the corrected function return the same object?

<details><summary>Expected result (reveal after predicting)</summary>

The faulty function changes the original list. The corrected function returns a new sorted list and leaves its input unchanged.
</details>

In [ ]:
def faulty_sort(durations):
    durations.sort()
    return durations


def sorted_copy(durations):
    return sorted(durations)


original_durations = [45, 15, 30]
alias = original_durations
faulty_sort(alias)
assert original_durations == [15, 30, 45]

original_durations = [45, 15, 30]
result = sorted_copy(original_durations)
assert result == [15, 30, 45]
assert original_durations == [45, 15, 30]
assert result is not original_durations

### 9.4 Debug a threshold comparison

**Concept.** `>` and `>=` differ only at the boundary, so ordinary examples can miss the defect. Make the rule explicit: a session qualifies if it is at least the minimum. Test one value below, equal to and above the threshold.

**Predict.** Which session disappears in the faulty filter? What should the corrected result be at 30 minutes?

<details><summary>Expected result (reveal after predicting)</summary>

The faulty `>` rule excludes the 30-minute session. The inclusive `>=` rule returns `[30, 31]`.
</details>

In [ ]:
def faulty_threshold_filter(durations, minimum):
    return [d for d in durations if d > minimum]


def inclusive_threshold_filter(durations, minimum):
    return [d for d in durations if d >= minimum]


boundary_data = [29, 30, 31]
print(faulty_threshold_filter(boundary_data, 30))
print(inclusive_threshold_filter(boundary_data, 30))
assert faulty_threshold_filter(boundary_data, 30) == [31]
assert inclusive_threshold_filter(boundary_data, 30) == [30, 31]

### 9.5 Debug an incorrect conversion factor

**Concept.** A conversion can produce plausible-looking numbers even when the factor is wrong. Thirty minutes is half an hour, so a useful verification point is `30 → 0.5`; test another value and an empty collection as well.

**Predict.** What will dividing by 100 return for 30 minutes? What is the correct hours conversion?

<details><summary>Expected result (reveal after predicting)</summary>

The faulty function returns `0.3` hours for 30 minutes. Dividing by 60 returns `0.5`, and 60 minutes becomes `1.0` hour.
</details>

In [ ]:
def faulty_hours_conversion(durations):
    return [duration / 100 for duration in durations]


def corrected_hours_conversion(durations):
    return [duration / 60 for duration in durations]


print(faulty_hours_conversion([30, 60]))
print(corrected_hours_conversion([30, 60]))
assert faulty_hours_conversion([30, 60]) == [0.3, 0.6]
assert corrected_hours_conversion([30, 60]) == [0.5, 1.0]
assert corrected_hours_conversion([]) == []

### 9.6 Debug a confusing interface

**Concept.** A signature with many positional settings is difficult to review and easy to misuse. Restricting the data argument to positional use and configuration to keyword-only use clarifies the call. A function's signature is part of its public design.

**Predict.** What does the keyword-only call report? Why are the invalid calls commented?

<details><summary>Expected result (reveal after predicting)</summary>

The valid call selects two sessions with an inclusive 30-minute minimum. The commented calls misuse the positional-only data argument or pass an option positionally.
</details>

In [ ]:
def clear_session_count(durations, /, *, minimum=30):
    return sum(duration >= minimum for duration in durations)


study_durations = [20, 30, 45]
print(clear_session_count(study_durations, minimum=30))
# Invalid: clear_session_count(durations=study_durations)
# Invalid: clear_session_count(study_durations, 30)
assert clear_session_count(study_durations, minimum=30) == 2
assert clear_session_count([], minimum=30) == 0

### 9.7 Verify the full contract with boundary cases

**Verification.** Lightweight `assert` statements make assumptions executable. Cover empty collections, one item, exact thresholds, zero variadic arguments, omitted metadata and repeated calls. These checks complement examples; they do not replace explanatory documentation.

**Reflect.** Which test would have caught the mutable default? Which would have caught the threshold comparison defect?

<details><summary>Expected result (reveal after predicting)</summary>

Every assertion passes. A repeated-call check catches shared mutable state; the equality-at-threshold check catches `>` in place of `>=`.
</details>

In [ ]:
assert calculate_total([]) == 0
assert calculate_total([12]) == 12
assert filter_sessions([29, 30, 31], 30) == [30, 31]
assert duration_statistics()["count"] == 0
assert report_for_topic("Testing") == ("Testing", 0, {})
assert build_duration_report("Testing")["durations"] == ()
assert build_metadata_report("Boundary")["metadata"] == {}
assert convert_to_hours([]) == []

first_call = collect_activity_label("Review")
second_call = collect_activity_label("Practice")
assert first_call == ["Review"]
assert second_call == ["Practice"]
assert first_call is not second_call
print("All boundary and repeated-call checks passed.")

## Session 7 close: reflect and transfer

1. How do parameters, arguments and argument binding differ?
2. Why are mutable defaults risky, and when is persistent state intentional?
3. When do `/` and `*` improve a function interface?
4. What do `*args` and `**kwargs` collect, and how does call-site unpacking differ?
5. Why should calculation functions return data rather than print it?
6. How do closures, `nonlocal` and loop-variable late binding work?
7. When is a lambda clearer than a named function?
8. How does decomposition make the Learning Assistant easier to test?

**Final verification:** run the notebook from top to bottom. Every code cell is self-contained with respect to the lesson's definitions, uses only standard-library behaviour and avoids interactive input. Assertions check key outputs, boundaries, and input preservation.
